# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [3]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [4]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [5]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [6]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [7]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [8]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

Similar to HW 3, conducting an initial two JOINs to test for correct output and allow for aggregation in the next step.

In [9]:
# 1. First join: Match 'citing' with 'PATENT' info
joined_citing = citations.join(
    patents.alias("citing_pat"), 
    col("citing") == col("citing_pat.PATENT")
)

# 2. Second join: Match 'cited' with 'PATENT' info
final_df = joined_citing.join(
    patents.alias("cited_pat"), 
    col("cited") == col("cited_pat.PATENT")
)

# 3. Select columns and alias them to your required layout
output_df = final_df.select(
    col("citing").alias("Citing"),
    col("citing_pat.POSTATE").alias("POstate_citing"),
    col("cited").alias("Cited"),
    col("cited_pat.POSTATE").alias("POstate_cited")
)

# 4. Display the formatted layout
output_df.show(10)


+-------+--------------+-------+-------------+
| Citing|POstate_citing|  Cited|POstate_cited|
+-------+--------------+-------+-------------+
|4483021|            MS|3070803|           IL|
|4133055|            NH|3070803|           IL|
|4253313|          NULL|3070803|           IL|
|5054122|          NULL|3070803|           IL|
|5557807|            FL|3070803|           IL|
|4484363|            CA|3070803|           IL|
|4921141|            CA|3070803|           IL|
|5469579|          NULL|3070803|           IL|
|5850636|            CA|3070803|           IL|
|4400830|            FL|3070805|           CA|
+-------+--------------+-------+-------------+
only showing top 10 rows



Now performing all steps in one script--The double join with the output above is generated, then is filtered to remove Null states and save only states with matching patents. Then the aggregated count is appended to the original patent dataframe, similar to SQL.

In [10]:
# 1. Double Join to tie states to both sides of the citation
joined_df = citations.join(
    patents.alias("citing_pat"), 
    col("citing") == col("citing_pat.PATENT")
).join(
    patents.alias("cited_pat"), 
    col("cited") == col("cited_pat.PATENT")
)

# 2. Filter out null states and isolate same-state matches
same_state_matches = joined_df.filter(
    col("citing_pat.POSTATE").isNotNull() & 
    col("cited_pat.POSTATE").isNotNull() & 
    (col("citing_pat.POSTATE") == col("cited_pat.POSTATE"))
)

# 3. Count up matches grouped by the Citing patent
counts_df = same_state_matches.groupBy(col("citing").alias("PATENT_ID")) \
                              .agg(count("*").alias("same_state_citations"))

# 4. Left join to attach the new column to ALL original patent columns
augmented_patents = patents.join(
    counts_df, 
    patents.PATENT == counts_df.PATENT_ID, 
    how="left"
).na.fill(value=0, subset=["same_state_citations"])

# 5. Order descending by our count column
top_10_patents = augmented_patents.orderBy(col("same_state_citations").desc())

# 6. Select ALL columns from the patents table plus the new calculated column
# This prevents selecting duplicate 'PATENT_ID' tracking columns from the join
top_10_patents.select(patents["*"], col("same_state_citations")).show(10)


+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|same_state_citations|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------------------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|                 125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|  